# Chicago Congestion — Exploratory Data Analysis

Covers: feature distributions, CRS checks, missing value audit, join quality.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sqlalchemy import text
from db import get_engine

engine = get_engine()

## 1. Road Segments

In [ ]:
roads = gpd.read_postgis(
    text('SELECT * FROM road_segments LIMIT 5000'),
    engine, geom_col='geometry'
)
print(f'Shape: {roads.shape}')
print(f'CRS: {roads.crs}')
roads.head()

In [ ]:
# Missing value audit
roads[['highway', 'lanes', 'maxspeed', 'length', 'oneway']].isnull().mean().rename('null_frac').to_frame()

In [ ]:
# Highway type distribution
roads['highway'].value_counts().head(15).plot(kind='barh', figsize=(8, 5), title='Highway type counts')
plt.tight_layout()

In [ ]:
# Segment length distribution (log scale)
roads['length'].plot(kind='hist', bins=60, figsize=(8, 4),
                     title='Segment length (m)', logy=True)
plt.tight_layout()

## 2. Traffic Counts

In [ ]:
counts = gpd.read_postgis(
    text('SELECT * FROM traffic_counts'),
    engine, geom_col='geometry'
)
print(f'Shape: {counts.shape}')
print(f'CRS: {counts.crs}')
counts.head()

In [ ]:
# Join quality
matched = counts['segment_id'].notna().sum()
print(f'{matched}/{len(counts)} points matched ({100*matched/len(counts):.1f}%)')

In [ ]:
# Volume distribution
vol_col = 'total_passing_vehicle_volume'
if vol_col in counts.columns:
    counts[vol_col].astype(float).plot(
        kind='hist', bins=50, figsize=(8, 4),
        title='Daily traffic volume distribution'
    )
    plt.tight_layout()

## 3. Feature Matrix (post-engineering)

In [ ]:
import os
feat_path = '../data/features.parquet'
if os.path.exists(feat_path):
    df = pd.read_parquet(feat_path)
    print(f'Shape: {df.shape}')
    display(df.describe())
else:
    print('features.parquet not yet generated — run src/features.py first')

In [ ]:
if os.path.exists(feat_path):
    df['congestion_score'].plot(
        kind='hist', bins=50, figsize=(8, 4),
        title='Congestion score distribution [0–1]'
    )
    plt.tight_layout()